# Linear Algebra Foundations Lab

## Geometry, transformations, tensors, bases, systems, and low-rank structure

This is a **guided teaching lab**, not a final assessment. It turns the full [linear-algebra prerequisite track](https://github.com/jjames/llm-wiki/tree/main/lessons/prerequisites/01-linear-algebra) into experiments you can inspect and modify.

**Recommended order:** prerequisite lessons → this notebook → Notebook 13 mastery lab  
**Time:** 90–120 minutes including investigations  
**Dependencies:** NumPy and Matplotlib only

### Learning goals

You will learn to:

1. interpret dot products, cosine similarity, and projection geometrically;
2. treat matrices as transformations and connect determinants to area scaling;
3. reason about tensor shapes, broadcasting, and Einstein notation;
4. solve linear systems, change coordinates, and diagnose ill-conditioning;
5. recognize least squares through residual orthogonality; and
6. use the SVD to expose rank, principal directions, and optimal low-rank approximations.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(18)


## 1. Vectors: length, alignment, and projection

For vectors $a,b\in\mathbb{R}^n$,

$$a\cdot b=\lVert a\rVert\lVert b\rVert\cos\theta.$$

The dot product combines magnitude and alignment. Cosine similarity removes magnitude. Projection separates $a$ into a component parallel to $b$ and a residual orthogonal to $b$.


In [ ]:
a = np.array([3.0, 2.0])
b = np.array([1.0, 2.0])

dot = a @ b
cosine = dot / (np.linalg.norm(a) * np.linalg.norm(b))
projection = (dot / (b @ b)) * b
residual = a - projection

assert abs(residual @ b) < 1e-12
assert abs(dot) <= np.linalg.norm(a) * np.linalg.norm(b) + 1e-12
np.testing.assert_allclose(projection + residual, a)

fig, ax = plt.subplots(figsize=(6, 5))
origin = np.zeros(2)
for vector, color, label in [
    (a, "C0", "a"), (b, "C1", "b"),
    (projection, "C2", "projection of a onto b"),
    (residual, "C3", "orthogonal residual"),
]:
    start = projection if label == "orthogonal residual" else origin
    ax.quiver(*start, *vector, angles="xy", scale_units="xy", scale=1, color=color, label=label)
ax.set(xlim=(-0.5, 4.5), ylim=(-0.5, 4.5), aspect="equal", title="Projection decomposes a into parallel + orthogonal parts")
ax.grid(alpha=0.25); ax.legend(loc="upper left")
plt.show()

print(f"dot={dot:.3f}; cosine={cosine:.3f}")
print("projection:", projection, "residual:", residual)


### Investigation 1

- Scale `a` by 10. Which of dot product and cosine similarity changes?
- Make `a` orthogonal to `b`, then parallel and antiparallel. Predict the cosine each time.
- Explain why the projection coefficient divides by $b\cdot b$ rather than $\lVert b\rVert$.


## 2. Matrices as transformations

A matrix is not merely a table. Multiplication $y=Ax$ maps vectors to vectors. Its columns are the images of the input basis vectors. Composition order matters: $BAx$ applies $A$ first and $B$ second.

In two dimensions, $|\det A|$ is the factor by which $A$ scales area; the sign records whether orientation flips.


In [ ]:
angles = np.linspace(0, 2 * np.pi, 300)
circle = np.stack([np.cos(angles), np.sin(angles)])
square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]], dtype=float)

theta = np.deg2rad(35)
rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
stretch = np.diag([2.0, 0.5])
shear = np.array([[1.0, 0.8], [0.0, 1.0]])
transform = shear @ rotation @ stretch

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, matrix, title in [(axes[0], np.eye(2), "Original"), (axes[1], transform, "Shear @ rotation @ stretch")]:
    ax.plot(*(matrix @ circle), color="C0")
    ax.plot(*(matrix @ square), color="C1", linewidth=2)
    ax.quiver([0, 0], [0, 0], matrix[0], matrix[1], angles="xy", scale_units="xy", scale=1, color=["C2", "C3"])
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", title=title)
    ax.grid(alpha=0.25)
plt.show()

np.testing.assert_allclose(np.linalg.det(transform), np.linalg.det(shear) * np.linalg.det(rotation) * np.linalg.det(stretch))
assert not np.allclose(shear @ rotation, rotation @ shear)
print(f"determinant={np.linalg.det(transform):.3f}; area scale={abs(np.linalg.det(transform)):.3f}")


## 3. Tensors, broadcasting, and `einsum`

In ML, an axis has meaning. A token batch might have shape `(batch, token, feature)`. A linear layer contracts the feature axis with a weight matrix. Broadcasting adds a bias across the batch and token axes without copying it conceptually.

Einstein notation names the axes that survive. The expression `btd,dh->bth` says: contract input feature `d`; retain batch `b`, token `t`, and output feature `h`.


In [ ]:
batch, tokens, d_model, d_hidden = 2, 4, 3, 5
representations = rng.normal(size=(batch, tokens, d_model))
weights = rng.normal(size=(d_model, d_hidden))
bias = rng.normal(size=(d_hidden,))

hidden_matmul = representations @ weights + bias
hidden_einsum = np.einsum("btd,dh->bth", representations, weights) + bias
scores = np.einsum("btd,bsd->bts", representations, representations)

assert hidden_matmul.shape == (batch, tokens, d_hidden)
assert scores.shape == (batch, tokens, tokens)
np.testing.assert_allclose(hidden_matmul, hidden_einsum)
np.testing.assert_allclose(scores, np.swapaxes(scores, 1, 2))

print("representations", representations.shape, "@ weights", weights.shape, "-> hidden", hidden_matmul.shape)
print("pairwise token scores:", scores.shape)


## 4. Systems, bases, conditioning, and least squares

Solving $Ax=b$ asks for coordinates $x$ whose column combination equals $b$. A basis matrix $B$ performs the same operation: $v=B[v]_B$.

When a square system is nearly singular, small perturbations in $b$ can produce large changes in $x$. The condition number quantifies worst-case sensitivity. For an overdetermined system, least squares chooses $\hat x$ so the residual $r=b-A\hat x$ is orthogonal to every column of $A$: $A^Tr=0$.


In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
target = np.array([7.0, 5.0])
solution = np.linalg.solve(A, target)
np.testing.assert_allclose(A @ solution, target)

basis = np.array([[1.0, 1.0], [1.0, -1.0]])
vector = np.array([4.0, 2.0])
coordinates = np.linalg.solve(basis, vector)
np.testing.assert_allclose(basis @ coordinates, vector)

design = np.column_stack([np.ones(8), np.linspace(-1, 1, 8)])
observations = np.array([1.0, 1.3, 1.8, 2.1, 2.8, 3.1, 3.7, 4.2])
coefficients, *_ = np.linalg.lstsq(design, observations, rcond=None)
residual_ls = observations - design @ coefficients
np.testing.assert_allclose(design.T @ residual_ls, np.zeros(2), atol=1e-12)

well = np.array([[1.0, 0.0], [0.0, 1.0]])
ill = np.array([[1.0, 1.0], [1.0, 1.000001]])
rhs = np.array([2.0, 2.000001])
perturbation = np.array([0.0, 1e-7])
well_change = np.linalg.norm(np.linalg.solve(well, rhs + perturbation) - np.linalg.solve(well, rhs))
ill_change = np.linalg.norm(np.linalg.solve(ill, rhs + perturbation) - np.linalg.solve(ill, rhs))
assert np.linalg.cond(ill) > 1e6
assert ill_change > 1e5 * well_change

print("system solution:", solution)
print("coordinates in new basis:", coordinates)
print("least-squares intercept/slope:", coefficients)
print(f"condition numbers: well={np.linalg.cond(well):.1f}, ill={np.linalg.cond(ill):.2e}")


## 5. Rank, SVD, and PCA

The singular value decomposition

$$X=U\Sigma V^T$$

expresses a matrix as orthonormal input directions, axis scaling, and orthonormal output directions. Keeping the largest $k$ singular values gives the best rank-$k$ approximation in Frobenius norm. PCA is the SVD of centered observations; its right singular vectors are principal directions in feature space.


In [ ]:
latent = rng.normal(size=(150, 2))
mixing = np.array([[2.0, 0.2, -1.0, 0.5], [0.3, 1.5, 0.8, -0.7]])
observations = latent @ mixing + 0.05 * rng.normal(size=(150, 4))
centered = observations - observations.mean(axis=0)
U, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)

reconstruction_errors = []
for rank in range(1, 5):
    approximation = (U[:, :rank] * singular_values[:rank]) @ Vt[:rank]
    reconstruction_errors.append(np.linalg.norm(centered - approximation, ord="fro"))

assert all(left >= right - 1e-12 for left, right in zip(reconstruction_errors, reconstruction_errors[1:]))
covariance = centered.T @ centered / (len(centered) - 1)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
principal_eigenvectors = eigenvectors[:, np.argsort(eigenvalues)[::-1]]
np.testing.assert_allclose(np.abs(principal_eigenvectors[:, :2].T @ Vt[:2].T), np.eye(2), atol=1e-6)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(1, 5), singular_values, color="C0")
axes[0].set(xlabel="component", ylabel="singular value", title="Two dominant latent directions")
axes[1].plot(range(1, 5), reconstruction_errors, marker="o", color="C1")
axes[1].set(xlabel="retained rank", ylabel="Frobenius error", title="Low-rank approximation error")
for ax in axes: ax.grid(alpha=0.25)
plt.show()

print("singular values:", singular_values)
print("reconstruction errors:", np.round(reconstruction_errors, 4))


## Cumulative mastery check

Without rerunning code, answer:

1. When is cosine similarity preferable to a dot product?
2. If $A$ has shape `(m, n)`, what spaces contain its columns, null vectors, and outputs?
3. Why does broadcasting a bias not introduce a separate parameter for every token?
4. What does a large condition number warn you about?
5. Why is a least-squares residual orthogonal to the design matrix's column space?
6. What do the first two right singular vectors mean in this experiment?

**Ready for Notebook 13 when:** you can predict shapes before execution, move between geometric and coordinate descriptions, and explain every assertion above. Then use Notebook 13 for a less guided representation-geometry audit.
